# Get env along the obs trajectory

In [1]:
cd ~/work/tutorials/sources/OpenDrift/openberg_july2026

/home/jovyan/work/tutorials/sources/OpenDrift/openberg_july2026


In [2]:
import glob
from src.utils import *
from openberg_july2026.src2.utils0 import *
from openberg_july2026.src2.utils2 import *

In [3]:
ib = 'iceberg2021w'
cmems_l = ['topaz4','topaz5', 'topaz6','glorys','windglophyre']
# cmems_l = ['topaz4','topaz5','topaz6','glorys','glophyanfcH','glophyanfcD','nextsimanfc','windglophynrt']
sim_l_contain, sim_l_notcontain = [ib],['idx',]

#### Preload and calc

#### Create file

In [4]:
out = xr.Dataset()
                # coords={'time_obs':obs.time.data,
                #         'cmems_id':[env_dict[cm] for cm in cmems_l],}
                         # 'time_sim':np.arange(dict_analysis_period[ib]['full'].start,dict_analysis_period[ib]['full'].stop,dtype='datetime64[h]'),
                         # 'time_sim':np.arange(obs_seed.time.data[0],obs_seed.time.data[-1]+
                                              # np.timedelta64(int(str(obs.seed_freq.data)[:-1]),str(obs.seed_freq.data)[-1]),dtype='datetime64[h]'),
                         # 'sim_id':[f.split('/')[-1].replace('.nc','') for f in file_l],
                         # 'sim_trajectory':np.arange(obs_seed.time.size*11)},
                # data_vars={v:(('time_obs','cmems_id'),np.full((obs.time.data.size,len(cmems_l)),np.nan)) for v in ['x_sea_water_velocity','y_sea_water_velocity',
                # "sea_ice_area_fraction","sea_ice_x_velocity","sea_ice_y_velocity"]}

#### Observations

In [5]:
#read obs
fname = 'merged_obs10'#'merged_obs_iceberg2026e_1D'
with xr.open_dataset('./input/%s.nc'%fname) as ds:
    obs = ds.sel(iceberg=ib,trajectory_id=ib) 
    obs_seed = obs.isel(time=(obs.seed_idx==1))

In [6]:
obs_sub_kin = compute_drift_kinematics(obs)
tl = [str(t)[:16] for t in obs.time.data]

In [7]:
#add obs velocity
out = out.assign_coords(time_obs=("time_obs", obs.time.data))

out['iceberg_u_ms'] = (('time_obs'), obs_sub_kin.u_ms.values)
out['iceberg_v_ms'] = (('time_obs'), obs_sub_kin.v_ms.values)
out['iceberg_speed_ms'] = (('time_obs'), obs_sub_kin.speed_ms.values)
out['iceberg_direction_deg'] = (('time_obs'), obs_sub_kin.direction_deg.values)

In [8]:
out

<xarray.Dataset> Size: 4kB
Dimensions:                (time_obs: 90)
Coordinates:
  * time_obs               (time_obs) datetime64[ns] 720B 2021-08-12T19:23:57...
Data variables:
    iceberg_u_ms           (time_obs) float64 720B nan -0.01478 ... -0.005406
    iceberg_v_ms           (time_obs) float64 720B nan 0.03578 ... -0.006041
    iceberg_speed_ms       (time_obs) float64 720B nan 0.03871 ... 0.008107
    iceberg_direction_deg  (time_obs) float64 720B nan 337.6 ... 285.0 221.8

#### Simulations

In [9]:
#simulation list
file_l = glob.glob("./results/%s/*.nc"%ib)
file_l = [f for f in file_l if all(ff in f for ff in sim_l_contain)]
file_l = [f for f in file_l if all(ff not in f for ff in sim_l_notcontain)]
print(file_l)

['./results/iceberg2021w/iceberg2021w_gebco_glorys_windglophyre_newcoefmix_joined.nc', './results/iceberg2021w/iceberg2021w_gebco_topaz4_windglophyre_newcoefmix_joined.nc', './results/iceberg2021w/iceberg2021w_gebco_topaz5_windglophyre_newcoefmix_joined.nc', './results/iceberg2021w/iceberg2021w_gebco_topaz6_topaz5_windglophyre_newcoefmix_joined.nc']


In [10]:
#load sim
try: #if not merged
    sim_l = [str(f.split('/')[-1][:-3]) for f in file_l] 
    simall = xr.open_mfdataset(file_l,combine='nested',concat_dim="run")
    simall = simall.assign_coords(run=sim_l) #if not merged yet
except:     #when merged lready
    simall = xr.open_mfdataset(file_l)
    sim_l = [str(f) for f in simall.run]

/tmp/ipykernel_29896/255881585.py:4: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  simall = xr.open_mfdataset(file_l,combine='nested',concat_dim="run")
/tmp/ipykernel_29896/255881585.py:4: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'trajectory' ('trajectory',) The recommendation is to set join explicitly for this case.
  simall = xr.open_mfdataset(file_l,combine='nested',concat_dim="run")


In [11]:
# add sim velocity
out = out.assign_coords(time_sim=("time_sim", simall.time.data))
out = out.assign_coords(sim_id=("sim_id", simall.run.data))
out = out.assign_coords(sim_trajectory=("sim_trajectory", simall.trajectory.data))

out['iceb_x_velocity'] = (('sim_id','sim_trajectory','time_sim'), simall.iceb_x_velocity.values)
out['iceb_y_velocity'] = (('sim_id','sim_trajectory','time_sim'), simall.iceb_y_velocity.values)

In [12]:
out

<xarray.Dataset> Size: 1MB
Dimensions:                (time_obs: 90, time_sim: 528, sim_id: 4,
                            sim_trajectory: 77)
Coordinates:
  * time_obs               (time_obs) datetime64[ns] 720B 2021-08-12T19:23:57...
  * time_sim               (time_sim) datetime64[ns] 4kB 2021-09-19T14:32:55....
  * sim_id                 (sim_id) <U63 1kB 'iceberg2021w_gebco_glorys_windg...
  * sim_trajectory         (sim_trajectory) int32 308B 33 34 35 ... 107 108 109
Data variables:
    iceberg_u_ms           (time_obs) float64 720B nan -0.01478 ... -0.005406
    iceberg_v_ms           (time_obs) float64 720B nan 0.03578 ... -0.006041
    iceberg_speed_ms       (time_obs) float64 720B nan 0.03871 ... 0.008107
    iceberg_direction_deg  (time_obs) float64 720B nan 337.6 ... 285.0 221.8
    iceb_x_velocity        (sim_id, sim_trajectory, time_sim) float32 650kB 0...
    iceb_y_velocity        (sim_id, sim_trajectory, time_sim) float32 650kB 0...

#### Cmems along obs

In [13]:
# add env along obs
out = out.assign_coords(cmems_id=("cmems_id", [env_dict[cm] for cm in cmems_l]))
    
for v in ['x_sea_water_velocity','y_sea_water_velocity',"sea_ice_area_fraction","sea_ice_x_velocity","sea_ice_y_velocity","x_wind","y_wind"]:
    out[v] = (('time_obs','cmems_id'), np.full((obs.time.data.size,len(cmems_l)),np.nan))


In [14]:
out

<xarray.Dataset> Size: 1MB
Dimensions:                (time_obs: 90, time_sim: 528, sim_id: 4,
                            sim_trajectory: 77, cmems_id: 5)
Coordinates:
  * time_obs               (time_obs) datetime64[ns] 720B 2021-08-12T19:23:57...
  * time_sim               (time_sim) datetime64[ns] 4kB 2021-09-19T14:32:55....
  * sim_id                 (sim_id) <U63 1kB 'iceberg2021w_gebco_glorys_windg...
  * sim_trajectory         (sim_trajectory) int32 308B 33 34 35 ... 107 108 109
  * cmems_id               (cmems_id) <U86 2kB 'cmems_mod_arc_phy_my_topaz4_P...
Data variables: (12/13)
    iceberg_u_ms           (time_obs) float64 720B nan -0.01478 ... -0.005406
    iceberg_v_ms           (time_obs) float64 720B nan 0.03578 ... -0.006041
    iceberg_speed_ms       (time_obs) float64 720B nan 0.03871 ... 0.008107
    iceberg_direction_deg  (time_obs) float64 720B nan 337.6 ... 285.0 221.8
    iceb_x_velocity        (sim_id, sim_trajectory, time_sim) float32 650kB 0...
    iceb_y_velocity        (sim_id, sim_trajectory, time_sim) float32 650kB 0...
    ...                     ...
    y_sea_water_velocity   (time_obs, cmems_id) float64 4kB nan nan ... nan nan
    sea_ice_area_fraction  (time_obs, cmems_id) float64 4kB nan nan ... nan nan
    sea_ice_x_velocity     (time_obs, cmems_id) float64 4kB nan nan ... nan nan
    sea_ice_y_velocity     (time_obs, cmems_id) float64 4kB nan nan ... nan nan
    x_wind                 (time_obs, cmems_id) float64 4kB nan nan ... nan nan
    y_wind                 (time_obs, cmems_id) float64 4kB nan nan ... nan nan

In [15]:
for cmems_id in cmems_l:
    
    if cmems_id in ['windglophyre','windglophynrt']:variables=['x_wind','y_wind']
    elif cmems_id in ['topaz6','glophyanfcH']: variables = ['x_sea_water_velocity','y_sea_water_velocity']
    elif cmems_id in ['nextsimre','nextsimanfc','glophyanfcD']: 
        variables = ["sea_ice_area_fraction",
                "sea_ice_x_velocity",
                "sea_ice_y_velocity"]
    else:
        variables = ['x_sea_water_velocity',
                'y_sea_water_velocity',
                "sea_ice_area_fraction",
                "sea_ice_x_velocity",
                "sea_ice_y_velocity"]
        
    cmems_id = env_dict[cmems_id]
    print(cmems_id,variables)
    
    if cmems_id.startswith('cmems_'):
    
        from opendrift.readers import reader_copernicusmarine
        reader = reader_copernicusmarine.Reader(
            dataset_id=cmems_id)
    else:
        from opendrift.readers import reader_netCDF_CF_generic
        reader = reader_netCDF_CF_generic.Reader(
            cmems_id)
        
    try:z0 = min(0.0, np.max(reader.z))
    except: z0=0
    
    for i,t in enumerate(tl):
        # print('%s/%s'%(i,len(tl)))
        env, profiles = reader.get_variables_interpolated(
            variables=variables,
            lon=obs.lon[i],
            lat=obs.lat[i],
            time=np.datetime64(t),
            z=z0
        )
        for v in variables:
            out[v].loc[{'time_obs':t,'cmems_id':cmems_id}] = env[v].data[0]

cmems_mod_arc_phy_my_topaz4_P1D-m ['x_sea_water_velocity', 'y_sea_water_velocity', 'sea_ice_area_fraction', 'sea_ice_x_velocity', 'sea_ice_y_velocity']


INFO - 2026-10-07T13:17:22Z - Selected dataset version: "202506"
INFO - 2026-10-07T13:17:22Z - Selected dataset part: "default"


cmems_mod_arc_phy_anfc_6km_detided_PT1H-i ['x_sea_water_velocity', 'y_sea_water_velocity', 'sea_ice_area_fraction', 'sea_ice_x_velocity', 'sea_ice_y_velocity']


INFO - 2026-10-07T13:17:38Z - Selected dataset version: "202311"
INFO - 2026-10-07T13:17:38Z - Selected dataset part: "default"
WARNING - 2026-10-07T13:17:38Z - You are using the dataset cmems_mod_arc_phy_anfc_6km_detided_PT1H-i, version '202311', part 'default'. This exact version and part of the dataset will be retired on the 2027-01-12T13:00:00.000Z. For more information you can check: https://marine.copernicus.eu/user-corner/product-roadmap/transition-information
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning


https://thredds.met.no/thredds/dodsC/cmems/topaz6/dataset-topaz6-arc-15min-3km-be.ncml ['x_sea_water_velocity', 'y_sea_water_velocity']


...

cmems_mod_glo_phy_my_0.083deg_P1D-m ['x_sea_water_velocity', 'y_sea_water_velocity', 'sea_ice_area_fraction', 'sea_ice_x_velocity', 'sea_ice_y_velocity']


INFO - 2026-10-07T13:19:34Z - Selected dataset version: "202311"
INFO - 2026-10-07T13:19:34Z - Selected dataset part: "default"
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - re

cmems_obs-wind_glo_phy_my_l4_0.125deg_PT1H ['x_sea_water_velocity', 'y_sea_water_velocity', 'sea_ice_area_fraction', 'sea_ice_x_velocity', 'sea_ice_y_velocity']


INFO - 2026-10-07T13:19:46Z - Selected dataset version: "202211"
INFO - 2026-10-07T13:19:46Z - Selected dataset part: "default"


AssertionError: ['x_sea_water_velocity', 'y_sea_water_velocity', 'sea_ice_area_fraction', 'sea_ice_x_velocity', 'sea_ice_y_velocity'] is not subset of ['air_density', 'surface_downward_eastward_stress', 'surface_downward_eastward_stress_bias', 'surface_downward_eastward_stress_standard_deviation_of_differences', 'eastward_wind', 'eastward_wind_bias', 'eastward_wind_standard_deviation_of_differences', 'surface_downward_northward_stress', 'surface_downward_northward_stress_bias', 'surface_downward_northward_stress_standard_deviation_of_differences', 'northward_wind', 'northward_wind_bias', 'northward_wind_standard_deviation_of_differences', 'number_of_observations', 'vertical_component_of_surface_downward_stress_curl', 'vertical_component_of_surface_downward_stress_curl_bias', 'vertical_component_of_surface_downward_stress_curl_difference_of_variances', 'divergence_of_surface_downward_stress', 'divergence_of_surface_downward_stress_bias', 'divergence_of_surface_downward_stress_difference

#### Checks

In [ ]:
out

In [ ]:
#sim spd
# for v in ['iceb_x_velocity','iceb_y_velocity']:
#     out.plot.scatter(x='time_sim',y=v)
#plt.show()
#obs spd
# for v in ['iceberg_u_ms','iceberg_v_ms','iceberg_speed_ms','iceberg_direction_deg']:
#     out.plot.scatter(x='time_obs',y=v)
# plt.show()
# env spd
# for v in ['x_sea_water_velocity','y_sea_water_velocity','sea_ice_x_velocity','sea_ice_y_velocity','sea_ice_area_fraction']:
#     if not all(np.isnan(out[v])): out.plot.scatter(x='time_obs',y=v)
# plt.show()

#### Save

In [ ]:
out.to_netcdf('./results/timeseries_coll/%s_timeseries_collection.nc'%ib)

In [ ]:
s = '0.25 & 0.27 & 0.31 & 0.22 & 0.23'
print(np.round(np.mean([float(ss) for ss in s.split(' & ')]),2))

In [ ]:
# s = '0.12 & 0.19 & 0.17 & 0.2 & 0.24'
s2 = 0.46
s=[float(ss) for ss in s.split(' & ')]
scalc = np.round([ss-s2 for ss in s],2)
print('%s & '*len(scalc)%tuple(scalc))
print(np.round(np.mean(scalc),2))